In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore



C:\Users\vsharma\AppData\Local\Temp\ipykernel_1960\35342877.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\vsharma\Desktop\GenAi\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
loader = PyPDFLoader("../data/Vineet.pdf")
loaded = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted = splitter.split_documents(loaded)

embedding = MistralAIEmbeddings(model="mistral-embed-2312")

In [4]:
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted,
    embedding = embedding
)

In [5]:
same_record = vector_store.similarity_search("Who is Patient")

print(same_record[0])

page_content='Laboratory Investigation Report
Patient Name : Mr. Vineet
Age/Gender : 22 Y 6 M 21 D / M
MaxID/Lab ID : ML05009859/4337032601346
Ref Doctor : Dr.Mukesh Chandra Jha
Centre : 5054 - Motherland Hospital
OP/IP No/UHID : OP/1019956/MLH/24/011848
Collection Date/Time : 31/Mar/2026 10:45AM
Reporting Date/Time : 31/Mar/2026 01:16PM
Clinical Pathology
Test Name Result Unit Bio Ref Interval
SIN No:B2B9465983
Booking Centre :5054 - Motherland Hospital, NH-01, Sector 119,, 9953777444
The authenticity of the report can be verified by scanning the Q R Code on top of the page
Page 3 of 8' metadata={'producer': 'PDFsharp 1.50.5147 (www.pdfsharp.com) (Original: HiQPdf 8.10)', 'creator': 'PDFsharp 1.50.5147 (www.pdfsharp.com)', 'creationdate': '', 'moddate': '2026-03-31T18:01:56+05:30', 'source': '../data/Vineet.pdf', 'total_pages': 8, 'page': 2, 'page_label': '3'}


In [6]:
from langchain.tools import tool

@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relavent data of the PDF documents, and these document have details about medical reports
    """

    print("Tool called")
    docs = vector_store.similarity_search(query = query, k=4)
    content = ""
    

    for data in docs:
        content = content + data.page_content + "\n"

    return content

In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = "openai/gpt-oss-20b")

In [8]:
System_Prompt = """
    You are a helpful assistant that answer question using retrieved context.
    ALWAYS use the `retriever_tool` tool for question required external knowledge
"""

In [9]:
from langchain.agents import create_agent

agents = create_agent(
    model = llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [10]:
query = "What is the name of patient, and what is the name of Doctor and what test it include ?"

response = agents.invoke({
    'messages':[{"role":"user", "content": query}]
})

result = response["messages"][-1].content
print(result)


Tool called
**Patient:** Mr. Vineet  
**Referring Doctor:** Dr. Mukesh Chandra Jha  

**Tests included in the report**

| Section | Tests / Parameters Mentioned |
|---------|------------------------------|
| **Clinical Pathology** | (The report lists a full panel of clinical‑pathology tests – the exact items are shown in the detailed test table on page 3 of the PDF.) |
| **Hematology** | • Erythrocyte Sedimentation Rate (ESR) <br>• (Other routine hematology parameters – e.g., CBC, differential, platelet count – are shown in the detailed hematology table on pages 5‑7 of the PDF.) |

So, the report covers a standard hematology panel (including ESR) and a broader clinical‑pathology panel, with the full list of individual test names available in the detailed tables of the PDF.
